# Real Data Pred Code

In [ ]:
import matplotlib.pyplot as plt
import pickle
import torch
import numpy as np

file_path = "/Volumes/SIMMAX/02-24-26 Vac/140-Vac1029pm.pkl" 

with open(file_path, "rb") as f:
    data = pickle.load(f)


img_array = np.array(data['cropped_data'][1000], dtype=np.float32)
img_tensor = torch.from_numpy(img_array).view(1, 1, 30, 640).to(DEVICE)

model.eval()

with torch.no_grad():
    x_pred, y_pred, int_norm_pred, exists_logit = model(img_tensor)

exists_prob = torch.sigmoid(exists_logit)[0]   

print(f"\nFile: {file_path}")
print(f"{'Slot':>5}  {'x_pred':>8}  {'y_pred':>7}  {'intensity':>12}  {'prob':>6}")
print("-" * 55)

plt.imshow(img_array, aspect='auto' , cmap='jet')

for k in range(model.n_max):
    p = exists_prob[k].item()
    
    if p > 0.2:
      
        px = x_pred[0, k].item() * IMG_W
        py = y_pred[0, k].item() * IMG_H
       
        norm_val = int_norm_pred[0, k].item()
        
        log_val  = (norm_val * LOG_I_RANGE) + LOG_I_MIN
        
        real_intensity = float(np.exp(log_val))
        
        print(f"  {k:3d}  {px:8.1f}  {py:7.1f}  {real_intensity:12.1f}  {p:6.3f}")


# Synthetic Data Pred

In [ ]:
import random

sample_idx = random.randint(0, len(val_ds) - 1)
img, target = val_ds[sample_idx]          

model.eval()

with torch.no_grad():
    x_pred, y_pred, int_norm_pred, exists_logit = model(img.unsqueeze(0).to(DEVICE))

exists_prob = torch.sigmoid(exists_logit)[0]   # (16,)

print(f"\nSample index: {sample_idx}")
print(f"{'Slot':>5}  {'x_pred':>8}  {'y_pred':>7}  {'intensity':>12}  {'prob':>6}")
print("-" * 50)

for k in range(model.n_max):
    p = exists_prob[k].item()
    if p > 0.4:
        x = x_pred[0, k].item() * IMG_W
        y = y_pred[0, k].item() * IMG_H
        
       
        norm_val = int_norm_pred[0, k].item()
        
        log_val  = (norm_val * LOG_I_RANGE) + LOG_I_MIN
        
        ii = float(np.exp(log_val))
        
        print(f"  {k:3d}  {x:8.1f}  {y:7.1f}  {ii:12.1f}  {p:6.3f}")

print("\n── Ground truth ──")
for k in range(N_MAX_BLOBS):
    if target[k, 3] > 0:
        gx = target[k, 0].item() * IMG_W
        gy = target[k, 1].item() * IMG_H
        gi = target[k, 2].item()  
        print(f"  {k:3d}  {gx:8.1f}  {gy:7.1f}  {gi:12.1f}")